# Notebook 2 — Community Detection and Removing Harry
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

In Notebook 1 you saw what the network measures: proximity. Not friendship, not importance, not who speaks to whom. Proximity.

Now we run an algorithm on top of that proximity data. The algorithm — Louvain community detection — has no knowledge of Hogwarts, the Order of the Phoenix, or the Malfoy family. It sees a graph with weights. It finds groups of nodes that are more connected to each other than to the rest of the graph.

**The question:** Does it find the factions Rowling built?

And then: **remove Harry entirely.** Every scene he dominates disappears from his weight on other characters. Who holds each world together when he's gone?

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import community as community_louvain
from collections import defaultdict
import time
import os

os.makedirs('images', exist_ok=True)

DATA = 'data/'
print('Ready.')

---
## Part 1 — EDA: The pre-computed community data

We pre-ran community detection for Books 3, 5, and 7 — with Harry and without Harry — and saved the results. Look at the structure before running anything yourself.

In [ ]:
ca = pd.read_csv(DATA + 'community_analysis.csv')

print(f'Shape: {ca.shape}')
print(f'Columns: {ca.columns.tolist()}')
print()
ca.head(8)

In [ ]:
for book in [3, 5, 7]:
    b = ca[ca['book'] == book]
    n_chars = len(b)
    n_comm_with = b['community_with_harry'].nunique()
    n_comm_without = b['community_without_harry'].nunique()
    print(f'Book {book}: {n_chars} characters | {n_comm_with} communities (with Harry) | {n_comm_without} communities (without Harry)')

In [ ]:
b5 = ca[ca['book'] == 5]
sizes = b5.groupby('community_with_harry').size().sort_values(ascending=False)

print('Book 5 community sizes (with Harry):')
print(sizes.to_string())
print(f'\nLargest community: {sizes.iloc[0]} characters')
print(f'Smallest: {sizes.iloc[-1]} characters')

In [ ]:
harry_rows = ca[ca['character'] == 'Harry Potter'][['book', 'community_with_harry', 'community_size_with_harry',
                                                      'community_without_harry', 'community_size_without_harry']]
print('Harry Potter across books:')
print(harry_rows.to_string(index=False))

---
## Part 2 — Before running: one prediction

The algorithm sees only edge weights. No character names. No plot. It finds groups of characters who appear together more than they appear with outsiders.

**Write one community you are confident Louvain will find in Book 3.** Name it, and name 2–3 characters you expect in it.

Just one. The one you're most certain about.

**Write here:**

*Community name:*

*Expected members:*

---
## Part 3 — Run Louvain on Book 3

In [ ]:
df3 = pd.read_csv(DATA + 'hp_book3_edges.csv')
G3 = nx.from_pandas_edgelist(df3, 'source', 'target', edge_attr='weight')
print(f'Book 3: {G3.number_of_nodes()} characters, {G3.number_of_edges()} pairs')

In [ ]:
t0 = time.time()
partition3 = community_louvain.best_partition(G3, weight='weight', random_state=42)
print(f'Done in {time.time()-t0:.3f}s')

n_communities = max(partition3.values()) + 1
print(f'Communities found: {n_communities}')

In [ ]:
degree3 = dict(G3.degree(weight='weight'))
comm_members3 = defaultdict(list)
for char, cid in partition3.items():
    comm_members3[cid].append(char)

for cid in sorted(comm_members3, key=lambda c: -len(comm_members3[c])):
    members = comm_members3[cid]
    if len(members) < 3:
        continue
    top = sorted(members, key=lambda n: degree3.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} characters):')
    print(f'  {" | ".join(top)}')
    print()

**Name each community.** For every group with 4+ members, write what HP world it represents.

Then find this: **the community containing Sirius, Lupin, Snape, and Pettigrew.**

The algorithm put them together from scene co-occurrence counts alone. It has never read the books. It doesn't know about the Marauders, or Azkaban, or what these four characters mean to each other.

It found the secret history of Book 3 from the structure of scenes.

**Fill in the table:**

| Community ID | Top members | What HP world is this? |
|---|---|---|
| | | |
| | | |
| | | |

**Write:** Did your predicted community appear? And — was the Marauder community on your list, or did the algorithm find something you didn't think to predict?

**Your answer:**

*Write here.*

---
## Part 4 — Louvain on Books 5 and 7

In [ ]:
df5 = pd.read_csv(DATA + 'hp_book5_edges.csv')
G5 = nx.from_pandas_edgelist(df5, 'source', 'target', edge_attr='weight')

t0 = time.time()
partition5 = community_louvain.best_partition(G5, weight='weight', random_state=42)
print(f'Book 5 — {G5.number_of_nodes()} characters | {max(partition5.values())+1} communities | {time.time()-t0:.3f}s')

degree5 = dict(G5.degree(weight='weight'))
comm_members5 = defaultdict(list)
for char, cid in partition5.items():
    comm_members5[cid].append(char)

print()
for cid in sorted(comm_members5, key=lambda c: -len(comm_members5[c])):
    members = comm_members5[cid]
    if len(members) < 4:
        continue
    top = sorted(members, key=lambda n: degree5.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} chars): {" | ".join(top)}')

harry_comm5 = partition5.get('Harry Potter')
print(f'\nHarry is in Community {harry_comm5} (size {len(comm_members5[harry_comm5])})')

In [ ]:
df7 = pd.read_csv(DATA + 'hp_book7_edges.csv')
G7 = nx.from_pandas_edgelist(df7, 'source', 'target', edge_attr='weight')

t0 = time.time()
partition7 = community_louvain.best_partition(G7, weight='weight', random_state=42)
print(f'Book 7 — {G7.number_of_nodes()} characters | {max(partition7.values())+1} communities | {time.time()-t0:.3f}s')

degree7 = dict(G7.degree(weight='weight'))
comm_members7 = defaultdict(list)
for char, cid in partition7.items():
    comm_members7[cid].append(char)

print()
for cid in sorted(comm_members7, key=lambda c: -len(comm_members7[c])):
    members = comm_members7[cid]
    if len(members) < 4:
        continue
    top = sorted(members, key=lambda n: degree7.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} chars): {" | ".join(top)}')

harry_comm7 = partition7.get('Harry Potter')
print(f'\nHarry is in Community {harry_comm7} (size {len(comm_members7[harry_comm7])})')

**Two things to find in Book 7:**
- Where are Snape and Dumbledore? Are they in the Horcrux hunt community — or somewhere else?
- Which community is Luna in? Who else is there?

**Fill in the table:**

| Book | Harry's community | Who is with him? | Who is NOT with him (that you expected)? |
|---|---|---|---|
| Book 3 | | | |
| Book 5 | | | |
| Book 7 | | | |

**Write:** Harry's community changes every book. In which book does the algorithm show him most separated from Ron and Hermione? Does that match how Book 5 felt to you as a reader?

**Your answer:**

*Write here.*

---
## Part 5 — Remove Harry

Harry is in nearly every scene. His presence floods the edges of every character he interacts with. When we remove him, the other characters' relationships become visible on their own terms — without Harry mediating everything.

**Before loading the results — write your predictions:**
- Without Harry, who leads the Order of the Phoenix world in Book 5? ___
- Without Harry, who leads the Ron/Hermione world? ___

In [ ]:
ca = pd.read_csv(DATA + 'community_analysis.csv')

print(ca[['book', 'character', 'community_without_harry', 'within_degree_without_harry']].head(10).to_string(index=False))

### Finding 1 — The Slytherin world: who really holds it together?

Global rank in Book 3: Draco is #8 overall. Goyle is #22. Crabbe is #21.

Remove Harry. Look at the Slytherin clique on its own terms.

In [ ]:
b3 = ca[ca['book'] == 3].copy()

draco_comm = b3[b3['character'] == 'Draco Malfoy']['community_without_harry'].values[0]

slytherin = b3[b3['community_without_harry'] == draco_comm].copy()
slytherin = slytherin.sort_values('within_degree_without_harry', ascending=False).reset_index(drop=True)
slytherin['within_rank'] = slytherin.index + 1

print(f'Slytherin community (Book 3, no Harry) — {len(slytherin)} characters')
print()
print(slytherin[['within_rank', 'character', 'global_degree', 'within_degree_without_harry']].head(8).to_string(index=False))

**Rowling named Draco the leader of his clique. The global ranking agrees — Draco is far above Goyle and Crabbe.**

Within the Slytherin community with Harry removed: who is #1?

Before you take a position — think about this: in which scenes does Rowling write "Draco" by name, and in which scenes does he become "he"? When Draco has a charged confrontation with Harry, she uses his name. When he's in a quiet corridor moment with his crew, she often doesn't. Now ask the same question about Goyle. Does that change what you think the algorithm was actually counting?

You must take a position:

> **Option A — The algorithm is right.** Rowling actually wrote Goyle as the structural center of that clique. She just presented Draco as the leader through dialogue and agency — the scene construction tells a different story.

> **Option B — The algorithm is wrong.** It counted explicit names, not leadership. Goyle is always "Goyle" because minor characters need naming; Draco becomes "he" in the scenes that matter. Explain what the edge count missed.

Write 3–4 sentences. Reference specific scenes from Book 3.

**Within-community ranking (Book 3, no Harry):**

1. 
2. 
3. 

**My position:**

*Write here.*

### Finding 2 — The Order world: Sirius or Dumbledore?

In [ ]:
b5 = ca[ca['book'] == 5].copy()

sirius_comm = b5[b5['character'] == 'Sirius Black']['community_without_harry'].values[0]

order_world = b5[b5['community_without_harry'] == sirius_comm].sort_values(
    'within_degree_without_harry', ascending=False).reset_index(drop=True)
order_world['within_rank'] = order_world.index + 1

print(f'Order world (Book 5, no Harry) — {len(order_world)} characters')
print()
print(order_world[['within_rank', 'character', 'within_degree_without_harry']].head(10).to_string(index=False))

print()
dumbledore_comm = b5[b5['character'] == 'Albus Dumbledore']['community_without_harry'].values[0]
print(f'Dumbledore is in community {dumbledore_comm}')
print(f'Sirius is in community {sirius_comm}')
print(f'Same community: {sirius_comm == dumbledore_comm}')

**Two different structural roles.**

Sirius is embedded in the Order community. He holds it together — the scenes at Grimmauld Place, the war council, the people who gather around the cause. When Harry is gone, Sirius becomes the gravitational center of that world.

Dumbledore is #2 in multiple communities. He never completely belongs to any single one.

**Write:** What kind of character is each of them, structurally? One is a pillar — deep in one world. The other is a connector — present across many worlds but embedded in none. Which is which — and does that match how Rowling wrote them, or does it surprise you?

**Your answer:**

*Write here.*

### Finding 3 — Hermione vs Ron: across every book

In [ ]:
print('Hermione vs Ron — within-community rank (without Harry)\n')

for book in [3, 5, 7]:
    b = ca[ca['book'] == book].copy()
    herm_comm = b[b['character'] == 'Hermione Granger']['community_without_harry'].values[0]
    
    community = b[b['community_without_harry'] == herm_comm].sort_values(
        'within_degree_without_harry', ascending=False).reset_index(drop=True)
    community['rank'] = community.index + 1
    
    for name in ['Hermione Granger', 'Ronald Weasley']:
        row = community[community['character'] == name]
        if len(row) > 0:
            r = int(row['rank'].values[0])
            wd = int(row['within_degree_without_harry'].values[0])
            print(f'Book {book} | {name:<22} within-rank #{r:<3} within-degree {wd}')
    print()

In [ ]:
books = [3, 5, 7]
herm_ranks, ron_ranks = [], []

for book in books:
    b = ca[ca['book'] == book].copy()
    herm_comm = b[b['character'] == 'Hermione Granger']['community_without_harry'].values[0]
    community = b[b['community_without_harry'] == herm_comm].sort_values(
        'within_degree_without_harry', ascending=False).reset_index(drop=True)
    community['rank'] = community.index + 1
    herm_ranks.append(community[community['character'] == 'Hermione Granger']['rank'].values[0])
    ron_ranks.append(community[community['character'] == 'Ronald Weasley']['rank'].values[0])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1, 2], herm_ranks, 'o-', color='purple', linewidth=2, markersize=10, label='Hermione')
ax.plot([0, 1, 2], ron_ranks, 's--', color='darkorange', linewidth=2, markersize=10, label='Ron')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Book 3', 'Book 5', 'Book 7'], fontsize=12)
ax.invert_yaxis()
ax.set_ylabel('Within-community rank (1 = highest)', fontsize=11)
ax.set_title('Hermione vs Ron — within-trio rank (no Harry)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/hermione_ron_rank.png', dpi=150)
plt.show()

**Hermione ranks above Ron within the trio community in every single book — Books 3, 5, and 7. Not once. Every time.**

This is the most arguable finding in the project. You must take a position and defend it with evidence from the books.

> **Option A — The data is right.** Rowling wrote Hermione as the structural glue of the trio. Without Harry, she holds the group together more than Ron does. The co-occurrence pattern reflects something real about whose presence organises the group's scenes.

> **Option B — The data is wrong.** What Ron provides to the trio is not captured by co-occurrence: emotional warmth, family, belonging, the sense of home. The algorithm measures scene proximity. Ron's role is measured by something the algorithm cannot see.

Write 4–5 sentences. Reference specific scenes or plot moments from any of the books.

**My position on Hermione > Ron:**

*Write here — take a side and argue it.*

---
## What to bring to Session 3

- Your Slytherin position (Option A or B) and your argument
- Your Hermione > Ron position and your argument
- The finding that surprised you most — and whether you think the algorithm was right or measuring the wrong thing

**Next:** Notebook 3 — Predictions and Refusals. The network will predict which characters Rowling should have put in scenes together. Some of those predicted meetings never happened — and the ones that didn't are the most interesting result in the project.